In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import cv2
import hashlib
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [4]:
import os

from google.colab import drive
drive.mount('/content/drive')

dataset_path = "/content/drive/My Drive/DSGP/Data"
print(os.listdir(dataset_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['ringworm', 'eczema', 'acne']


In [5]:
import os
import pandas as pd

classes = sorted(os.listdir(dataset_path))

data = []
for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    for img in os.listdir(cls_path):
        data.append([os.path.join(cls_path, img), cls])

df = pd.DataFrame(data, columns=["image_path", "label"])

print(df["label"].value_counts())

label
ringworm    1000
eczema       208
Name: count, dtype: int64


In [6]:
import matplotlib.pyplot as plt
from PIL import Image
import random

plt.figure(figsize=(10,6))
for i, cls in enumerate(classes):
    img_path = random.choice(df[df["label"] == cls]["image_path"].values)
    img = Image.open(img_path)
    plt.subplot(1, 3, i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")
plt.show()

IndexError: Cannot choose from an empty sequence

<Figure size 1000x600 with 0 Axes>

In [ ]:
sizes = []

for path in df["image_path"]:
    img = Image.open(path)
    sizes.append(img.size)

pd.Series(sizes).value_counts().head()

In [ ]:
brightness = []
for path in df["image_path"]:
    img = Image.open(path).convert("L")
    brightness.append(np.array(img).mean())

pd.Series(brightness).describe()

In [ ]:
#Remove Corrupted Images
def remove_corrupted_images(dataset_path):
    removed = 0
    for cls in os.listdir(dataset_path):
        cls_path = os.path.join(dataset_path, cls)
        for img in os.listdir(cls_path):
            try:
                Image.open(os.path.join(cls_path, img)).verify()
            except:
                os.remove(os.path.join(cls_path, img))
                removed += 1
    print("Removed corrupted images:", removed)

remove_corrupted_images(dataset_path)

In [ ]:
aspect_ratios = []

for path in df["image_path"]:
    img = Image.open(path)
    w, h = img.size
    aspect_ratios.append(round(w / h, 2))

pd.Series(aspect_ratios).value_counts().head()

In [ ]:
import numpy as np

brightness = []

for path in df["image_path"]:
    img = Image.open(path).convert("L")  # grayscale
    brightness.append(np.array(img).mean())

pd.Series(brightness).describe()

In [ ]:
r, g, b = [], [], []

for path in df["image_path"]:
    img = Image.open(path).resize((64,64))
    arr = np.array(img)
    r.append(arr[:,:,0].mean())
    g.append(arr[:,:,1].mean())
    b.append(arr[:,:,2].mean())

print("R mean:", np.mean(r))
print("G mean:", np.mean(g))
print("B mean:", np.mean(b))

In [ ]:
def convert_to_rgb(dataset_path):
    for cls in os.listdir(dataset_path):
        cls_path = os.path.join(dataset_path, cls)
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            img = Image.open(img_path)
            if img.mode != "RGB":
                img.convert("RGB").save(img_path)

convert_to_rgb(dataset_path)

In [ ]:
hashes = {}
for path in df["image_path"]:
    h = hashlib.md5(open(path,'rb').read()).hexdigest()
    if h in hashes:
        os.remove(path)
    else:
        hashes[h] = path

In [ ]:
import hashlib

def image_hash(image_path):
    with open(image_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = {}
duplicates = []

for path in df["image_path"]:
    h = image_hash(path)
    if h in hashes:
        duplicates.append(path)
    else:
        hashes[h] = path

print("Duplicate images found:", len(duplicates))


In [ ]:
MIN_SIZE = 100  # pixels

for path in df["image_path"]:
    img = Image.open(path)
    w, h = img.size
    if w < MIN_SIZE or h < MIN_SIZE:
        os.remove(path)

In [ ]:
data = []
for cls in classes:
    for img in os.listdir(os.path.join(dataset_path, cls)):
        data.append([os.path.join(dataset_path, cls, img), cls])

df = pd.DataFrame(data, columns=["image_path", "label"])

In [ ]:
import cv2

def apply_clahe(img_path):
    img = cv2.imread(img_path)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))
    final = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    return final

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["label_encoded"].values)
)

train_ds = train_ds.map(load_image).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)


In [ ]:
#Remove non-image files

valid_ext = (".jpg", ".jpeg", ".png")

for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    for file in os.listdir(cls_path):
        if not file.lower().endswith(valid_ext):
            os.remove(os.path.join(cls_path, file))


In [ ]:
label_map = {cls: i for i, cls in enumerate(classes)}
df["label_encoded"] = df["label"].map(label_map)

print(label_map)

In [ ]:
data = []
for cls in classes:
    cls_path = os.path.join(dataset_path, cls)
    for img in os.listdir(cls_path):
        data.append([os.path.join(cls_path, img), cls])

df = pd.DataFrame(data, columns=["image_path", "label"])
df["label_encoded"] = df["label"].map(label_map)

In [ ]:
img = Image.open(df["image_path"].iloc[0]).resize((224,224))
arr = np.array(img) / 255.0
print(arr.min(), arr.max())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label_encoded"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_encoded"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
# PREPROCESSING (FOR CNN) Image generators (Rescale + Augmentation)

import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

In [ ]:
train_data = train_gen.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_data = val_gen.flow_from_dataframe(
    val_df,
    x_col="image_path",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_data = val_gen.flow_from_dataframe(
    test_df,
    x_col="image_path",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["label_encoded"].values)
)

train_ds = train_ds.map(load_image).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["label_encoded"]),
    y=df["label_encoded"]
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

In [ ]:
images, labels = next(train_data)
print(images.shape)
print(labels.shape)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
])

In [ ]:
inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)